# Custom CNN — Brain Tumor MRI Classification

**Model Owner:** Weerakkody W.W.M.B  
**IT Number:** IT25101776  
**Assigned Model:** Custom CNN

This notebook contains **two separately trained variants** of the assigned model.  
The better variant is selected using **Validation Macro F1**, and the Official Test set is used only after variant selection.

## 1. Project, Member and Model Overview

**Problem:** four-class Brain Tumor MRI classification.

**Classes:** glioma, meningioma, no-tumor and pituitary (the exact folder names are detected automatically).

**Assigned model:** **Custom CNN**

**Final selection rule:** choose the best tuned variety using **Validation Macro F1**, then evaluate that selected variety on the untouched Official Test set.

## 2. Model Suitability and Final Evaluation Requirements


### Final Evaluation coverage

This notebook is organized around the implementation-viva requirements:

- **Student:** Weerakkody W.W.M.B — IT25101776
- **Assigned model:** Custom CNN
- **Method:** training from scratch with no pretrained weights
- **Model suitability:** A Custom CNN is suitable because it learns MRI image features directly and its depth can be controlled for this relatively small four-class dataset. It also provides a clear from-scratch baseline.
- **Implementation details:** TensorFlow/Keras, model builder, augmentation, optimizer, learning rate, dropout and callbacks are shown in code.
- **Manual parameter tuning:** Variant A and Variant B use different augmentation strength, classifier size, dropout and learning-rate settings.
- **Variety comparison:** both varieties are evaluated using Validation Accuracy, Macro Precision, Macro Recall and Macro F1.
- **Validation method:** Training learns weights; Validation selects the optimum variety; Official Test is used only after selection.
- **Conclusion:** the variety with the highest Validation Macro F1 is selected.
- **Limitations/improvements:** included in Topic 19.
- **Demonstration output:** learning curves, tables, confusion matrix, per-class metrics, CSVs and a saved `.keras` model are produced.

## 3. Install Required Libraries

The notebook uses TensorFlow/Keras for deep learning, KaggleHub for dataset access, scikit-learn for evaluation metrics, pandas for result tables and matplotlib for learning curves.

In [ ]:
%pip -q install kagglehub scikit-learn

## 4. Import Libraries and Set Reproducibility

In [ ]:
from pathlib import Path
import json
import shutil
import zipfile
import random
import gc
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

import kagglehub

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

KAGGLE_DATASET = "maleeshaweerakkody/brain-tumor-mri-preprocessed"

print("TensorFlow version:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))
# ------------------------------------------------------------
# Model ownership
# ------------------------------------------------------------
MODEL_OWNER = "Weerakkody W.W.M.B"
OWNER_IT_NUMBER = "IT25101776"
OWNER_MEMBER_NO = OWNER_MEMBER_NO

print("Model Owner:", MODEL_OWNER)
print("IT Number:", OWNER_IT_NUMBER)
print("Assigned Model:", "Custom CNN")


## 5. Download the Final Preprocessed Kaggle Dataset

The final group dataset is downloaded directly from KaggleHub. No Google Drive path is required.

In [ ]:
KAGGLE_ROOT = Path(
    kagglehub.dataset_download(
        KAGGLE_DATASET
    )
)

print("Downloaded dataset:")
print(KAGGLE_ROOT)

WORK = Path("/content/brain_tumor_final_evaluation")
DATA_ROOT = WORK / "dataset"
MODELS_DIR = WORK / "models"
RESULTS_DIR = WORK / "results"

for p in [
    WORK,
    MODELS_DIR,
    RESULTS_DIR
]:
    p.mkdir(
        parents=True,
        exist_ok=True
    )

if DATA_ROOT.exists():
    shutil.rmtree(
        DATA_ROOT
    )

DATA_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

zip_files = list(
    KAGGLE_ROOT.rglob("*.zip")
)

if zip_files:
    print("ZIP files found:")

    for zip_path in zip_files:
        print("-", zip_path)

        with zipfile.ZipFile(
            zip_path,
            "r"
        ) as z:
            z.extractall(
                DATA_ROOT
            )

    SEARCH_ROOT = DATA_ROOT

else:
    print(
        "Dataset already contains extracted folders."
    )

    SEARCH_ROOT = KAGGLE_ROOT

## 6. Detect Training, Validation and Official Test Folders

In [ ]:
def find_split_dir(
    root,
    possible_names
):
    names = {
        x.lower()
        for x in possible_names
    }

    matches = [
        p
        for p in root.rglob("*")
        if p.is_dir()
        and p.name.lower() in names
    ]

    if not matches:
        return None

    matches = sorted(
        matches,
        key=lambda p: (
            -len([
                x
                for x in p.iterdir()
                if x.is_dir()
            ]),
            len(p.parts)
        )
    )

    return matches[0]


TRAIN_DIR = find_split_dir(
    SEARCH_ROOT,
    ["train", "training"]
)

VAL_DIR = find_split_dir(
    SEARCH_ROOT,
    ["validation", "valid", "val"]
)

TEST_DIR = find_split_dir(
    SEARCH_ROOT,
    ["test", "testing"]
)

print("Training   :", TRAIN_DIR)
print("Validation :", VAL_DIR)
print("Testing    :", TEST_DIR)

assert TRAIN_DIR is not None, (
    "Training folder was not found."
)

assert VAL_DIR is not None, (
    "Validation folder was not found."
)

assert TEST_DIR is not None, (
    "Official Test folder was not found."
)

## 7. Exploratory Data Analysis and Class Distribution

This section checks the number of images in every class and split before model training.

In [ ]:
IMAGE_EXTS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".tif",
    ".tiff"
}


def count_images(
    root
):
    rows = []

    for class_dir in sorted(
        [
            p
            for p in root.iterdir()
            if p.is_dir()
        ]
    ):
        count = sum(
            1
            for p in class_dir.rglob("*")
            if p.is_file()
            and p.suffix.lower()
            in IMAGE_EXTS
        )

        rows.append({
            "Class": class_dir.name,
            "Count": count
        })

    return pd.DataFrame(
        rows
    )


train_counts = count_images(
    TRAIN_DIR
).rename(
    columns={
        "Count": "Training"
    }
)

val_counts = count_images(
    VAL_DIR
).rename(
    columns={
        "Count": "Validation"
    }
)

test_counts = count_images(
    TEST_DIR
).rename(
    columns={
        "Count": "Testing"
    }
)

class_counts = (
    train_counts
    .merge(
        val_counts,
        on="Class",
        how="outer"
    )
    .merge(
        test_counts,
        on="Class",
        how="outer"
    )
    .fillna(0)
)

display(
    class_counts
)

print(
    "Training total:",
    int(
        class_counts[
            "Training"
        ].sum()
    )
)

print(
    "Validation total:",
    int(
        class_counts[
            "Validation"
        ].sum()
    )
)

print(
    "Testing total:",
    int(
        class_counts[
            "Testing"
        ].sum()
    )
)

class_counts.set_index(
    "Class"
)[
    [
        "Training",
        "Validation",
        "Testing"
    ]
].plot(
    kind="bar",
    figsize=(10, 5)
)

plt.title(
    "Brain Tumor MRI — Class Distribution"
)

plt.xlabel("Class")
plt.ylabel("Number of Images")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## 8. Build TensorFlow Datasets and Verify Exact Class Order

The class order is saved and later exported to `best_model_metadata.json`. This is important for correct localhost predictions.

In [ ]:
IMG_SIZE = (
    224,
    224
)

BATCH_SIZE = 16

train_ds = (
    tf.keras.utils
    .image_dataset_from_directory(
        TRAIN_DIR,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        label_mode="categorical",
        shuffle=True,
        seed=SEED
    )
)

CLASS_NAMES = (
    train_ds.class_names
)

NUM_CLASSES = len(
    CLASS_NAMES
)

val_ds = (
    tf.keras.utils
    .image_dataset_from_directory(
        VAL_DIR,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        label_mode="categorical",
        shuffle=False
    )
)

test_ds = (
    tf.keras.utils
    .image_dataset_from_directory(
        TEST_DIR,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        label_mode="categorical",
        shuffle=False
    )
)

validation_classes = sorted(
    [
        p.name
        for p in VAL_DIR.iterdir()
        if p.is_dir()
    ]
)

test_classes = sorted(
    [
        p.name
        for p in TEST_DIR.iterdir()
        if p.is_dir()
    ]
)

print(
    "Training classes  :",
    CLASS_NAMES
)

print(
    "Validation classes:",
    validation_classes
)

print(
    "Testing classes   :",
    test_classes
)

assert (
    CLASS_NAMES
    == validation_classes
), "Validation class order mismatch."

assert (
    CLASS_NAMES
    == test_classes
), "Testing class order mismatch."

AUTOTUNE = (
    tf.data.AUTOTUNE
)

train_ds = train_ds.prefetch(
    AUTOTUNE
)

val_ds = val_ds.prefetch(
    AUTOTUNE
)

test_ds = test_ds.prefetch(
    AUTOTUNE
)

print(
    "Class-order verification: PASSED"
)

## 9. Preprocessing and Training-Only Augmentation Strategy

### 9.1 Where this notebook starts in the full preprocessing pipeline

```text
Original raw MRI dataset
        ↓
Group preprocessing completed before model training
        ↓
Duplicate cleaning
Leakage-safe Train / Validation / Test preparation
Resize / padding to 224×224
CLAHE enhancement
        ↓
Final Kaggle preprocessed dataset
        ↓
THIS NOTEBOOK STARTS HERE
        ↓
Load 224×224 RGB MRI images
        ↓
Training-only augmentation
        ↓
Custom CNN-specific preprocessing
        ↓
Custom CNN
```

The final Kaggle dataset is already cleaned and image-preprocessed, so **CLAHE, duplicate removal and leakage removal are not repeated inside the model notebook**.

### 9.2 Custom CNN-specific preprocessing


**Model-specific preprocessing used here:** `layers.Rescaling(1.0 / 255.0)`

The final Kaggle images arrive in the normal image range of approximately `0–255`.
The Custom CNN is trained from scratch, so the notebook explicitly converts them to `0–1`.


### 9.3 Training-only augmentation

Random flip, rotation, translation, zoom and contrast are used only while Training. Validation and Official Test images are not randomly augmented.

This separation is important: **group preprocessing prepares the dataset once; model-specific preprocessing prepares the tensor for the selected architecture.**

In [ ]:
# ============================================================
# MODEL-SPECIFIC PREPROCESSING DEMONSTRATION
# Model: Custom CNN
# Mode : scratch
# ============================================================

PREPROCESS_DEMO_IMAGES, PREPROCESS_DEMO_LABELS = next(
    iter(train_ds)
)

PREPROCESS_DEMO_SAMPLE = PREPROCESS_DEMO_IMAGES[:4]

print("Model:", "Custom CNN")
print("Mode :", "scratch")
print("Input shape:", PREPROCESS_DEMO_SAMPLE.shape)
print(
    "Input pixel range:",
    float(tf.reduce_min(PREPROCESS_DEMO_SAMPLE)),
    "to",
    float(tf.reduce_max(PREPROCESS_DEMO_SAMPLE))
)

PREPROCESS_DEMO_OUTPUT = layers.Rescaling(
    1.0 / 255.0
)(
    PREPROCESS_DEMO_SAMPLE
)

print(
    "After model-specific preprocessing:",
    float(tf.reduce_min(PREPROCESS_DEMO_OUTPUT)),
    "to",
    float(tf.reduce_max(PREPROCESS_DEMO_OUTPUT))
)

print("Transformation: Rescaling(1/255)")

# Show the same input MRI images used for this preprocessing demonstration.
plt.figure(figsize=(10, 5))

for i in range(min(4, PREPROCESS_DEMO_SAMPLE.shape[0])):
    plt.subplot(1, 4, i + 1)
    plt.imshow(
        PREPROCESS_DEMO_SAMPLE[i]
        .numpy()
        .astype(np.uint8)
    )

    class_index = int(
        np.argmax(
            PREPROCESS_DEMO_LABELS[i]
            .numpy()
        )
    )

    plt.title(
        CLASS_NAMES[class_index]
    )

    plt.axis("off")

plt.suptitle(
    "Final Group-Preprocessed MRI Inputs Before Model-Specific Transform"
)

plt.tight_layout()
plt.show()

In [ ]:
def make_augmentation(
    variant_name
):
    # Variant A = lighter augmentation.
    # Variant B = slightly stronger but still moderate augmentation.
    if variant_name == "A":
        rotation = 0.02
        translation = 0.02
        zoom = 0.03
        contrast = 0.05

    else:
        rotation = 0.04
        translation = 0.04
        zoom = 0.05
        contrast = 0.08

    return keras.Sequential(
        [
            layers.RandomFlip(
                "horizontal"
            ),

            layers.RandomRotation(
                rotation
            ),

            layers.RandomTranslation(
                translation,
                translation
            ),

            layers.RandomZoom(
                zoom
            ),

            layers.RandomContrast(
                contrast
            ),
        ],
        name=(
            f"augmentation_"
            f"{variant_name}"
        )
    )


print(
    "Variant A: lighter training-only augmentation"
)

print(
    "Variant B: moderate training-only augmentation"
)

## 10. Evaluation Metrics and Helper Functions

Metrics used:
- Accuracy
- Macro Precision
- Macro Recall
- Macro F1
- Per-class Precision / Recall / F1
- Confusion Matrix

**Macro F1 is the main model-selection metric.**

In [ ]:
def collect_true_labels(
    ds
):
    labels = []

    for _, y_batch in ds:
        labels.append(
            np.argmax(
                y_batch.numpy(),
                axis=1
            )
        )

    return np.concatenate(
        labels
    )


def evaluate_model(
    model,
    ds,
    y_true
):
    probabilities = (
        model.predict(
            ds,
            verbose=0
        )
    )

    predictions = (
        np.argmax(
            probabilities,
            axis=1
        )
    )

    metrics = {
        "accuracy": float(
            accuracy_score(
                y_true,
                predictions
            )
        ),

        "precision_macro": float(
            precision_score(
                y_true,
                predictions,
                average="macro",
                zero_division=0
            )
        ),

        "recall_macro": float(
            recall_score(
                y_true,
                predictions,
                average="macro",
                zero_division=0
            )
        ),

        "f1_macro": float(
            f1_score(
                y_true,
                predictions,
                average="macro",
                zero_division=0
            )
        ),
    }

    return (
        metrics,
        predictions,
        probabilities
    )


Y_VAL_TRUE = collect_true_labels(
    val_ds
)

Y_TEST_TRUE = collect_true_labels(
    test_ds
)


def new_macro_f1():
    return tf.keras.metrics.F1Score(
        average="macro",
        name="f1_macro"
    )

## 11. Manual Hyperparameter Tuning Plan — Variant A vs Variant B

Two varieties are trained to satisfy parameter-tuning and model-variety requirements.

- **Variant A — Baseline CNN:** Dense=128, Dropout=0.3, Initial LR=0.001, Fine-tune last=N/A, Fine-tune LR=N/A, Glioma weight=1.0
- **Variant B — Deeper Regularized CNN:** Dense=256, Dropout=0.4, Initial LR=0.0003, Fine-tune last=N/A, Fine-tune LR=N/A, Glioma weight=1.0

The two variants also use slightly different augmentation strengths.

### Descriptive variant names

- **Variant A:** Baseline CNN
- **Variant B:** Deeper Regularized CNN

These names are used in training output, graph titles, comparison tables, CSV files and final selected-model output.


## 12. Build the Two Model Varieties

In [ ]:
MODEL_NAME = "Custom CNN"
MODEL_SLUG = "custom_cnn"
MEMBER_NO = OWNER_MEMBER_NO
PRETRAINED = False

VARIANT_NAMES = {
    "A": "Baseline CNN",
    "B": "Deeper Regularized CNN",
}

VARIANTS = {'A': {'filters': [32, 64, 128], 'dense': 128, 'dropout': 0.3, 'learning_rate': 0.001, 'batch_norm': True, 'glioma_weight': 1.0}, 'B': {'filters': [32, 64, 128, 256], 'dense': 256, 'dropout': 0.4, 'learning_rate': 0.0003, 'batch_norm': True, 'glioma_weight': 1.0}}


def build_variant(
    variant_name
):
    config = VARIANTS[
        variant_name
    ]

    inputs = keras.Input(
        shape=(
            224,
            224,
            3
        )
    )

    x = make_augmentation(
        variant_name
    )(
        inputs
    )

    x = layers.Rescaling(
        1.0 / 255.0
    )(x)

    for filters in config[
        "filters"
    ]:
        x = layers.Conv2D(
            filters,
            3,
            padding="same",
            use_bias=False
        )(x)

        if config[
            "batch_norm"
        ]:
            x = layers.BatchNormalization()(x)

        x = layers.ReLU()(x)

        x = layers.MaxPooling2D()(x)

    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dense(
        config[
            "dense"
        ],
        activation="relu"
    )(x)

    x = layers.Dropout(
        config[
            "dropout"
        ]
    )(x)

    outputs = layers.Dense(
        NUM_CLASSES,
        activation="softmax"
    )(x)

    model = keras.Model(
        inputs,
        outputs,
        name=(
            f"Custom_CNN_"
            f"Variant_{variant_name}"
        )
    )

    model.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=config[
                "learning_rate"
            ]
        ),
        loss=keras.losses.CategoricalCrossentropy(
            label_smoothing=0.05
        ),
        metrics=[
            "accuracy",
            new_macro_f1()
        ]
    )

    return (
        model,
        None,
        config
    )

## 13. Train Variant A Separately — Baseline CNN

Variant A is trained in its **own cell**.  
Its best checkpoint is selected only from Validation Macro F1.

In [ ]:
def make_class_weight(
    config
):
    class_weight = {
        i: 1.0
        for i in range(
            NUM_CLASSES
        )
    }

    glioma_weight = float(
        config.get(
            "glioma_weight",
            1.0
        )
    )

    if glioma_weight != 1.0:
        matches = [
            i
            for i, name
            in enumerate(
                CLASS_NAMES
            )
            if "glioma"
            in name.lower()
        ]

        if matches:
            class_weight[
                matches[0]
            ] = glioma_weight

    return class_weight


def history_to_frame(
    history,
    phase_name
):
    df = pd.DataFrame(
        history.history
    )

    df[
        "Phase"
    ] = phase_name

    return df


def train_single_variant(
    variant_name
):
    variant_display_name = VARIANT_NAMES[
        variant_name
    ]

    print(
        "\n"
        + "=" * 75
    )

    print(
        f"TRAINING {MODEL_NAME} "
        f"VARIANT {variant_name} — "
        f"{variant_display_name}"
    )

    print(
        "=" * 75
    )

    tf.keras.backend.clear_session()
    gc.collect()

    model, base_model, config = build_variant(
        variant_name
    )

    class_weight = make_class_weight(
        config
    )

    print(
        "Class weight:",
        class_weight
    )

    phase1_path = (
        MODELS_DIR
        / (
            f"member{MEMBER_NO}_"
            f"{MODEL_SLUG}_"
            f"{variant_name}_phase1.keras"
        )
    )

    phase2_path = (
        MODELS_DIR
        / (
            f"member{MEMBER_NO}_"
            f"{MODEL_SLUG}_"
            f"{variant_name}_phase2.keras"
        )
    )

    if PRETRAINED:
        phase1_callbacks = [
            keras.callbacks.ModelCheckpoint(
                phase1_path,
                monitor="val_f1_macro",
                mode="max",
                save_best_only=True,
                verbose=1
            ),

            keras.callbacks.EarlyStopping(
                monitor="val_f1_macro",
                mode="max",
                patience=5,
                restore_best_weights=True,
                verbose=1
            ),

            keras.callbacks.ReduceLROnPlateau(
                monitor="val_loss",
                mode="min",
                factor=0.5,
                patience=2,
                min_lr=1e-7,
                verbose=1
            )
        ]

        history_phase1 = model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=15,
            callbacks=phase1_callbacks,
            class_weight=class_weight,
            verbose=1
        )

        phase1_best = (
            keras.models.load_model(
                phase1_path
            )
        )

        phase1_metrics, _, _ = evaluate_model(
            phase1_best,
            val_ds,
            Y_VAL_TRUE
        )

        base_model.trainable = True

        fine_tune_last = int(
            config[
                "fine_tune_last"
            ]
        )

        for layer in base_model.layers[
            :-fine_tune_last
        ]:
            layer.trainable = False

        for layer in base_model.layers[
            -fine_tune_last:
        ]:
            if isinstance(
                layer,
                layers.BatchNormalization
            ):
                layer.trainable = False
            else:
                layer.trainable = True

        model.compile(
            optimizer=keras.optimizers.Adam(
                learning_rate=config[
                    "fine_tune_lr"
                ]
            ),
            loss=keras.losses.CategoricalCrossentropy(
                label_smoothing=0.05
            ),
            metrics=[
                "accuracy",
                new_macro_f1()
            ]
        )

        phase2_callbacks = [
            keras.callbacks.ModelCheckpoint(
                phase2_path,
                monitor="val_f1_macro",
                mode="max",
                save_best_only=True,
                verbose=1
            ),

            keras.callbacks.EarlyStopping(
                monitor="val_f1_macro",
                mode="max",
                patience=6,
                restore_best_weights=True,
                verbose=1
            ),

            keras.callbacks.ReduceLROnPlateau(
                monitor="val_loss",
                mode="min",
                factor=0.5,
                patience=2,
                min_lr=1e-7,
                verbose=1
            )
        ]

        history_phase2 = model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=25,
            callbacks=phase2_callbacks,
            class_weight=class_weight,
            verbose=1
        )

        phase2_best = (
            keras.models.load_model(
                phase2_path
            )
        )

        phase2_metrics, _, _ = evaluate_model(
            phase2_best,
            val_ds,
            Y_VAL_TRUE
        )

        if (
            phase2_metrics[
                "f1_macro"
            ]
            >=
            phase1_metrics[
                "f1_macro"
            ]
        ):
            chosen_path = phase2_path
            chosen_metrics = phase2_metrics
            chosen_phase = "Fine-tuned Phase 2"
        else:
            chosen_path = phase1_path
            chosen_metrics = phase1_metrics
            chosen_phase = "Frozen-backbone Phase 1"

        history_df = pd.concat(
            [
                history_to_frame(
                    history_phase1,
                    "Phase 1"
                ),
                history_to_frame(
                    history_phase2,
                    "Phase 2"
                )
            ],
            ignore_index=True
        )

    else:
        callbacks = [
            keras.callbacks.ModelCheckpoint(
                phase1_path,
                monitor="val_f1_macro",
                mode="max",
                save_best_only=True,
                verbose=1
            ),

            keras.callbacks.EarlyStopping(
                monitor="val_f1_macro",
                mode="max",
                patience=7,
                restore_best_weights=True,
                verbose=1
            ),

            keras.callbacks.ReduceLROnPlateau(
                monitor="val_loss",
                mode="min",
                factor=0.5,
                patience=3,
                min_lr=1e-6,
                verbose=1
            )
        ]

        history = model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=35,
            callbacks=callbacks,
            class_weight=class_weight,
            verbose=1
        )

        chosen_path = phase1_path

        chosen_model = (
            keras.models.load_model(
                chosen_path
            )
        )

        chosen_metrics, _, _ = evaluate_model(
            chosen_model,
            val_ds,
            Y_VAL_TRUE
        )

        chosen_phase = "From Scratch"

        history_df = history_to_frame(
            history,
            "Scratch"
        )

    result = {
        "Variant": variant_name,
        "Variant Name": variant_display_name,
        "Selected Phase": chosen_phase,
        "Dense Units": config[
            "dense"
        ],
        "Dropout": config[
            "dropout"
        ],
        "Initial LR": config[
            "learning_rate"
        ],
        "Fine Tune Last": (
            config.get(
                "fine_tune_last",
                "N/A"
            )
            if PRETRAINED
            else "N/A"
        ),
        "Fine Tune LR": (
            config.get(
                "fine_tune_lr",
                "N/A"
            )
            if PRETRAINED
            else "N/A"
        ),
        "Glioma Weight": config.get(
            "glioma_weight",
            1.0
        ),
        "Validation Accuracy": chosen_metrics[
            "accuracy"
        ],
        "Validation Precision": chosen_metrics[
            "precision_macro"
        ],
        "Validation Recall": chosen_metrics[
            "recall_macro"
        ],
        "Validation F1": chosen_metrics[
            "f1_macro"
        ],
        "_checkpoint": chosen_path,
        "_history": history_df,
        "_config": config,
    }

    return result


# ------------------------------------------------------------
# TRAIN VARIANT A ONLY IN THIS CELL
# ------------------------------------------------------------

VARIANT_A_RESULT = train_single_variant(
    "A"
)

print(
    f"\nVariant A — {VARIANT_NAMES['A']} finished."
)

display(
    pd.DataFrame([
        {
            k: v
            for k, v in VARIANT_A_RESULT.items()
            if not k.startswith("_")
        }
    ])
)

## 14. Variant A Visualization — Baseline CNN — Accuracy, Loss and Macro F1

This section visualizes Variant A before Variant B is trained.

In [ ]:
def plot_single_variant(
    result,
    title
):
    history_df = result[
        "_history"
    ]

    epochs = np.arange(
        1,
        len(history_df) + 1
    )

    plt.figure(
        figsize=(8, 4)
    )

    plt.plot(
        epochs,
        history_df[
            "accuracy"
        ],
        label="Training Accuracy"
    )

    plt.plot(
        epochs,
        history_df[
            "val_accuracy"
        ],
        label="Validation Accuracy"
    )

    plt.title(
        f"{title} — Accuracy"
    )

    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.tight_layout()
    plt.show()


    plt.figure(
        figsize=(8, 4)
    )

    plt.plot(
        epochs,
        history_df[
            "loss"
        ],
        label="Training Loss"
    )

    plt.plot(
        epochs,
        history_df[
            "val_loss"
        ],
        label="Validation Loss"
    )

    plt.title(
        f"{title} — Loss"
    )

    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.tight_layout()
    plt.show()


    if (
        "f1_macro"
        in history_df.columns
        and
        "val_f1_macro"
        in history_df.columns
    ):
        plt.figure(
            figsize=(8, 4)
        )

        plt.plot(
            epochs,
            history_df[
                "f1_macro"
            ],
            label="Training Macro F1"
        )

        plt.plot(
            epochs,
            history_df[
                "val_f1_macro"
            ],
            label="Validation Macro F1"
        )

        plt.title(
            f"{title} — Macro F1"
        )

        plt.xlabel("Epoch")
        plt.ylabel("Macro F1")
        plt.legend()
        plt.tight_layout()
        plt.show()


plot_single_variant(
    VARIANT_A_RESULT,
    f"{MODEL_NAME} Variant A — {VARIANT_NAMES['A']}"
)

print(
    f"Variant A — {VARIANT_NAMES['A']} Validation Summary"
)

display(
    pd.DataFrame([
        {
            k: v
            for k, v in VARIANT_A_RESULT.items()
            if not k.startswith("_")
        }
    ])
)

## 15. Train Variant B Separately — Deeper Regularized CNN

Variant B is trained in a **different cell** from Variant A.  
This makes the two experiments easier to explain and demonstrate in the viva.

In [ ]:
# ------------------------------------------------------------
# TRAIN VARIANT B ONLY IN THIS CELL
# ------------------------------------------------------------

VARIANT_B_RESULT = train_single_variant(
    "B"
)

print(
    f"\nVariant B — {VARIANT_NAMES['B']} finished."
)

display(
    pd.DataFrame([
        {
            k: v
            for k, v in VARIANT_B_RESULT.items()
            if not k.startswith("_")
        }
    ])
)

## 16. Variant B Visualization — Deeper Regularized CNN — Accuracy, Loss and Macro F1

This section shows Variant B's learning behaviour independently.

In [ ]:
plot_single_variant(
    VARIANT_B_RESULT,
    f"{MODEL_NAME} Variant B — {VARIANT_NAMES['B']}"
)

print(
    f"Variant B — {VARIANT_NAMES['B']} Validation Summary"
)

display(
    pd.DataFrame([
        {
            k: v
            for k, v in VARIANT_B_RESULT.items()
            if not k.startswith("_")
        }
    ])
)

## 17. Compare Variant A (Baseline CNN) vs Variant B (Deeper Regularized CNN) and Select the Best

Only after both variants are trained and visualized separately are their Validation results compared.

The winner is the variant with the highest **Validation Macro F1**.

In [ ]:
VARIANT_RESULTS = [
    VARIANT_A_RESULT,
    VARIANT_B_RESULT
]

VARIANT_ROWS = [
    {
        k: v
        for k, v in result.items()
        if not k.startswith("_")
    }
    for result in VARIANT_RESULTS
]

variant_results = pd.DataFrame(
    VARIANT_ROWS
).sort_values(
    "Validation F1",
    ascending=False
).reset_index(
    drop=True
)

variant_results[
    "Variant Label"
] = variant_results[
    "Variant"
].map(
    lambda v: (
        f"{v} — "
        f"{VARIANT_NAMES[v]}"
    )
)

print(
    "VARIANT A vs VARIANT B — FINAL VALIDATION COMPARISON"
)

print(
    f"A = {VARIANT_NAMES['A']}"
)

print(
    f"B = {VARIANT_NAMES['B']}"
)

display(
    variant_results
)

comparison_metrics = [
    "Validation Accuracy",
    "Validation Precision",
    "Validation Recall",
    "Validation F1"
]

for metric_name in comparison_metrics:
    plt.figure(
        figsize=(7, 4)
    )

    plt.bar(
        variant_results[
            "Variant Label"
        ],
        variant_results[
            metric_name
        ]
    )

    plt.title(
        f"{MODEL_NAME} — "
        f"{metric_name}"
    )

    plt.xlabel("Variant")
    plt.ylabel(metric_name)
    plt.tight_layout()
    plt.show()


BEST_VARIANT = str(
    variant_results.loc[
        0,
        "Variant"
    ]
)

if BEST_VARIANT == "A":
    BEST_RESULT = VARIANT_A_RESULT
else:
    BEST_RESULT = VARIANT_B_RESULT

BEST_VARIANT_NAME = VARIANT_NAMES[
    BEST_VARIANT
]

BEST_CHECKPOINT = (
    BEST_RESULT[
        "_checkpoint"
    ]
)

FINAL_MODEL_PATH = (
    MODELS_DIR
    / (
        f"member{MEMBER_NO}_"
        f"selected_{MODEL_SLUG}.keras"
    )
)

shutil.copy2(
    BEST_CHECKPOINT,
    FINAL_MODEL_PATH
)

VARIANT_HISTORIES = {
    "A": VARIANT_A_RESULT[
        "_history"
    ],
    "B": VARIANT_B_RESULT[
        "_history"
    ],
}

VARIANT_PATHS = {
    "A": VARIANT_A_RESULT[
        "_checkpoint"
    ],
    "B": VARIANT_B_RESULT[
        "_checkpoint"
    ],
}

print(
    "\nSELECTED VARIANT:",
    BEST_VARIANT,
    "—",
    BEST_VARIANT_NAME
)

print(
    "Selection rule:",
    "Highest Validation Macro F1"
)

print(
    "Selected model path:",
    FINAL_MODEL_PATH
)

## 18. Official Test Evaluation, Error Analysis and Result Export

The selected variant is now evaluated on the Official Test set.

This section includes:
- Validation and Test metrics,
- Classification Report,
- Confusion Matrix,
- per-class Precision / Recall / F1,
- glioma → meningioma diagnostic,
- final member result CSV,
- variant comparison CSV,
- history CSV files,
- exact `best_model_metadata.json`.

In [ ]:
best_model = (
    keras.models.load_model(
        FINAL_MODEL_PATH
    )
)

val_metrics, val_pred, val_probs = evaluate_model(
    best_model,
    val_ds,
    Y_VAL_TRUE
)

test_metrics, test_pred, test_probs = evaluate_model(
    best_model,
    test_ds,
    Y_TEST_TRUE
)

print(
    "SELECTED VARIANT:",
    BEST_VARIANT,
    "—",
    BEST_VARIANT_NAME
)

print(
    "\nVALIDATION METRICS"
)

display(
    pd.DataFrame([
        val_metrics
    ])
)

print(
    "\nOFFICIAL TEST METRICS"
)

display(
    pd.DataFrame([
        test_metrics
    ])
)

report_dict = classification_report(
    Y_TEST_TRUE,
    test_pred,
    target_names=CLASS_NAMES,
    zero_division=0,
    output_dict=True
)

classification_report_df = (
    pd.DataFrame(
        report_dict
    )
    .transpose()
)

print(
    "\nOFFICIAL TEST CLASSIFICATION REPORT"
)

display(
    classification_report_df
)

cm = confusion_matrix(
    Y_TEST_TRUE,
    test_pred
)

confusion_matrix_df = pd.DataFrame(
    cm,
    index=[
        f"True {x}"
        for x in CLASS_NAMES
    ],
    columns=[
        f"Pred {x}"
        for x in CLASS_NAMES
    ]
)

display(
    confusion_matrix_df
)

plt.figure(
    figsize=(8, 7)
)

plt.imshow(
    cm
)

plt.title(
    f"{MODEL_NAME} — Official Test Confusion Matrix"
)

plt.xlabel("Predicted Class")
plt.ylabel("True Class")

plt.xticks(
    np.arange(
        NUM_CLASSES
    ),
    CLASS_NAMES,
    rotation=45,
    ha="right"
)

plt.yticks(
    np.arange(
        NUM_CLASSES
    ),
    CLASS_NAMES
)

for i in range(
    cm.shape[0]
):
    for j in range(
        cm.shape[1]
    ):
        plt.text(
            j,
            i,
            str(
                cm[i, j]
            ),
            ha="center",
            va="center"
        )

plt.tight_layout()
plt.show()

per_class_rows = []

for class_name in CLASS_NAMES:
    class_info = (
        report_dict[
            class_name
        ]
    )

    per_class_rows.append({
        "Class": class_name,
        "Precision": float(
            class_info[
                "precision"
            ]
        ),
        "Recall": float(
            class_info[
                "recall"
            ]
        ),
        "F1": float(
            class_info[
                "f1-score"
            ]
        ),
        "Support": int(
            class_info[
                "support"
            ]
        ),
    })

per_class_df = pd.DataFrame(
    per_class_rows
)

display(
    per_class_df
)


plt.figure(
    figsize=(9, 4)
)

plt.bar(
    per_class_df[
        "Class"
    ],
    per_class_df[
        "Recall"
    ]
)

plt.title(
    f"{MODEL_NAME} — Per-Class Test Recall"
)

plt.ylabel("Recall")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


plt.figure(
    figsize=(9, 4)
)

plt.bar(
    per_class_df[
        "Class"
    ],
    per_class_df[
        "F1"
    ]
)

plt.title(
    f"{MODEL_NAME} — Per-Class Test F1"
)

plt.ylabel("F1")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


glioma_matches = [
    i
    for i, name in enumerate(
        CLASS_NAMES
    )
    if "glioma"
    in name.lower()
]

meningioma_matches = [
    i
    for i, name in enumerate(
        CLASS_NAMES
    )
    if "meningioma"
    in name.lower()
]

if (
    glioma_matches
    and meningioma_matches
):
    glioma_index = (
        glioma_matches[0]
    )

    meningioma_index = (
        meningioma_matches[0]
    )

    glioma_total = int(
        cm[
            glioma_index
        ].sum()
    )

    glioma_to_meningioma = int(
        cm[
            glioma_index,
            meningioma_index
        ]
    )

    glioma_recall = float(
        report_dict[
            CLASS_NAMES[
                glioma_index
            ]
        ][
            "recall"
        ]
    )

    print(
        "Glioma Recall:",
        f"{glioma_recall * 100:.2f}%"
    )

    print(
        "Glioma → Meningioma errors:",
        glioma_to_meningioma,
        "/",
        glioma_total
    )

member_final_result = pd.DataFrame([
    {
        "Member": OWNER_MEMBER_NO,
        "Student": MODEL_OWNER,
        "IT Number": OWNER_IT_NUMBER,
        "Model": "Custom CNN",
        "Weights": "None - trained from scratch",
        "Selected Variant": BEST_VARIANT,
        "Selected Variant Name": BEST_VARIANT_NAME,

        "Validation F1": float(
            val_metrics[
                "f1_macro"
            ]
        ),

        "Test Accuracy": float(
            test_metrics[
                "accuracy"
            ]
        ),

        "Test Precision": float(
            test_metrics[
                "precision_macro"
            ]
        ),

        "Test Recall": float(
            test_metrics[
                "recall_macro"
            ]
        ),

        "Test F1": float(
            test_metrics[
                "f1_macro"
            ]
        ),
    }
])

display(
    member_final_result
)

MEMBER_RESULT_CSV = (
    RESULTS_DIR
    / (
        "it25101776_"
        "custom_cnn_final_result.csv"
    )
)

VARIANT_CSV = (
    RESULTS_DIR
    / (
        "it25101776_"
        "custom_cnn_variant_comparison.csv"
    )
)

REPORT_CSV = (
    RESULTS_DIR
    / (
        "it25101776_"
        "custom_cnn_classification_report.csv"
    )
)

CM_CSV = (
    RESULTS_DIR
    / (
        "it25101776_"
        "custom_cnn_confusion_matrix.csv"
    )
)

PER_CLASS_CSV = (
    RESULTS_DIR
    / (
        "it25101776_"
        "custom_cnn_per_class_metrics.csv"
    )
)

HISTORY_A_CSV = (
    RESULTS_DIR
    / (
        "it25101776_"
        "custom_cnn_history_A.csv"
    )
)

HISTORY_B_CSV = (
    RESULTS_DIR
    / (
        "it25101776_"
        "custom_cnn_history_B.csv"
    )
)

member_final_result.to_csv(
    MEMBER_RESULT_CSV,
    index=False
)

variant_results.to_csv(
    VARIANT_CSV,
    index=False
)

classification_report_df.to_csv(
    REPORT_CSV
)

confusion_matrix_df.to_csv(
    CM_CSV
)

per_class_df.to_csv(
    PER_CLASS_CSV,
    index=False
)

VARIANT_HISTORIES[
    "A"
].to_csv(
    HISTORY_A_CSV,
    index=False
)

VARIANT_HISTORIES[
    "B"
].to_csv(
    HISTORY_B_CSV,
    index=False
)


metadata = {
    "model_owner": MODEL_OWNER,
    "owner_it_number": OWNER_IT_NUMBER,
    "owner_member_no": OWNER_MEMBER_NO,
    "member": OWNER_MEMBER_NO,
    "student": MODEL_OWNER,
    "it_number": OWNER_IT_NUMBER,
    "selected_model": "Custom CNN",
    "weights": "None - trained from scratch",
    "pretrained": False,
    "selected_variant": BEST_VARIANT,
    "selected_variant_name": BEST_VARIANT_NAME,
    "variant_names": VARIANT_NAMES,
    "selection_metric": (
        "Validation Macro F1"
    ),
    "checkpoint_monitor": (
        "val_f1_macro"
    ),
    "class_names": CLASS_NAMES,
    "dataset": KAGGLE_DATASET,
    "image_size": [
        224,
        224,
        3
    ],
    "external_dataset_preprocessing": (
        "Final dataset is already resized/padded to 224x224 "
        "and CLAHE-enhanced by the group preprocessing pipeline."
    ),
    "model_specific_preprocessing_is_saved_in_model": True,
    "variant_configs": VARIANTS,
    "validation_metrics": val_metrics,
    "test_metrics": test_metrics,
    "class_weight_note": (
        "EfficientNetB0 ImageNet Variant B tests a mild glioma "
        "weight of 1.40 because the previous model showed "
        "glioma-to-meningioma confusion; other variants use 1.0."
    ),
}

METADATA_PATH = (
    RESULTS_DIR
    / "best_model_metadata.json"
)

with open(
    METADATA_PATH,
    "w"
) as f:
    json.dump(
        metadata,
        f,
        indent=2
    )

print(
    "Saved member result:",
    MEMBER_RESULT_CSV
)

print(
    "Saved variant comparison:",
    VARIANT_CSV
)

print(
    "Saved classification report:",
    REPORT_CSV
)

print(
    "Saved confusion matrix:",
    CM_CSV
)

print(
    "Saved per-class metrics:",
    PER_CLASS_CSV
)

print(
    "Saved metadata:",
    METADATA_PATH
)

## 19. Download the Selected Keras Model, CSV Files and Viva Package

The notebook downloads the selected `.keras` model and the most important result files. It also creates one ZIP containing all related outputs.


### Viva-ready explanation

**Why is this model suitable?**  
A Custom CNN is suitable because it learns MRI image features directly and its depth can be controlled for this relatively small four-class dataset. It also provides a clear from-scratch baseline.

**Why are two varieties trained?**  
The final evaluation asks for parameter tuning and comparison across different varieties. Variant A (**Baseline CNN**) and Variant B (**Deeper Regularized CNN**) change augmentation and hyperparameters. The better variety is selected using Validation Macro F1.

**Why Validation Macro F1?**  
Macro F1 gives equal importance to glioma, meningioma, no-tumor and pituitary classes. A model should not look good only because it performs very well on one or two classes.

**Why is the Official Test set not used to choose the variety?**  
Using the Test set for tuning would leak information into model selection. Training learns parameters, Validation selects the model, and Test reports final generalization.

**Main limitation:**  
Training a CNN from scratch on a relatively small MRI dataset can overfit and normally needs more real data than transfer learning.

**Possible improvements:**
- collect more real MRI scans from diverse sources,
- perform more systematic hyperparameter tuning,
- inspect misclassified glioma/meningioma images,
- consider cross-validation when computational resources allow,
- calibrate model confidence,
- validate on an independent external dataset.

**Academic-use note:**  
This model is for an academic AIML project and is not a clinical diagnostic system.

### Preprocessing viva sentence

> “The dataset was already group-preprocessed before model training. In this notebook I then apply the preprocessing required by **Custom CNN (scratch)**, and I keep random augmentation only in the Training stage.”


### Variant names to remember

- **Variant A:** Baseline CNN
- **Variant B:** Deeper Regularized CNN


In [ ]:
from google.colab import files

PACKAGE_DIR = (
    WORK
    / "it25101776_final_package"
)

if PACKAGE_DIR.exists():
    shutil.rmtree(
        PACKAGE_DIR
    )

PACKAGE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

PACKAGE_MODEL = (
    PACKAGE_DIR
    / (
        "Member1_"
        "custom_cnn_SELECTED.keras"
    )
)

shutil.copy2(
    FINAL_MODEL_PATH,
    PACKAGE_MODEL
)

related_files = [
    MEMBER_RESULT_CSV,
    VARIANT_CSV,
    REPORT_CSV,
    CM_CSV,
    PER_CLASS_CSV,
    HISTORY_A_CSV,
    HISTORY_B_CSV,
    METADATA_PATH,
]

for src in related_files:
    shutil.copy2(
        src,
        PACKAGE_DIR
        / Path(
            src
        ).name
    )

PACKAGE_ZIP = Path(
    shutil.make_archive(
        (
            "/content/"
            "Member1_"
            "custom_cnn_FINAL_PACKAGE"
        ),
        "zip",
        root_dir=PACKAGE_DIR
    )
)

print(
    "Downloading selected .keras model..."
)

files.download(
    str(
        PACKAGE_MODEL
    )
)

print(
    "Downloading final member comparison CSV..."
)

files.download(
    str(
        MEMBER_RESULT_CSV
    )
)

print(
    "Downloading variant comparison CSV..."
)

files.download(
    str(
        VARIANT_CSV
    )
)

print(
    "Downloading metadata JSON..."
)

files.download(
    str(
        METADATA_PATH
    )
)

print(
    "Downloading complete results ZIP..."
)

files.download(
    str(
        PACKAGE_ZIP
    )
)